# Controls

> Gallery control components for view toggle and type filtering.

In [ ]:
#| default_exp components.controls

In [ ]:
#| export
from typing import Any, List, Optional

from fasthtml.common import Div, Button, Span

# Tailwind utilities
from cjm_fasthtml_tailwind.utilities.spacing import p, m
from cjm_fasthtml_tailwind.utilities.flexbox_and_grid import (
    flex_display, items, justify, flex_wrap, gap
)
from cjm_fasthtml_tailwind.utilities.borders import rounded
from cjm_fasthtml_tailwind.core.base import combine_classes

# DaisyUI utilities
from cjm_fasthtml_daisyui.components.actions.button import (
    btn, btn_colors, btn_sizes, btn_styles
)
from cjm_fasthtml_daisyui.components.data_display.badge import badge, badge_colors, badge_styles
from cjm_fasthtml_daisyui.utilities.semantic_colors import bg_dui, text_dui
from cjm_fasthtml_daisyui.components.navigation.pagination import join, join_item

# Local imports
from cjm_file_discovery.core.models import FileType
from cjm_fasthtml_media_gallery.core.config import GalleryConfig, ViewMode
from cjm_fasthtml_media_gallery.core.icons import (
    get_gallery_icon, get_media_type_icon, MEDIA_TYPE_BADGE_COLORS
)

## View Mode Toggle

Buttons to switch between grid and list view.

In [ ]:
#| export
def render_view_toggle(
    current_mode: ViewMode,           # Current view mode
    toggle_url: str,                  # URL for toggle handler
    hx_target: Optional[str] = None,  # HTMX target for swap
) -> Any:  # View toggle button group
    """Render view mode toggle buttons."""
    def make_button(mode: ViewMode, icon_key: str, label: str) -> Button:
        is_active = current_mode == mode
        
        btn_cls = combine_classes(
            btn, btn_sizes.sm, join_item,
            btn_colors.primary if is_active else btn_styles.ghost
        )
        
        attrs = {
            "cls": btn_cls,
            "title": label,
            "aria-label": label,
        }
        
        if not is_active:
            attrs["hx_post"] = toggle_url
            attrs["hx_vals"] = f'{{"view_mode": "{mode.value}"}}'
            if hx_target:
                attrs["hx_target"] = hx_target
            attrs["hx_swap"] = "outerHTML"
        
        return Button(
            get_gallery_icon(icon_key, size=4),
            **attrs
        )
    
    return Div(
        make_button(ViewMode.GRID, "grid_view", "Grid view"),
        make_button(ViewMode.LIST, "list_view", "List view"),
        cls=str(join)
    )

In [ ]:
from fasthtml.common import to_xml

# Test view toggle - grid mode active
toggle = render_view_toggle(ViewMode.GRID, "/toggle-view", "#gallery")
html = to_xml(toggle)
assert "join" in html
assert "btn-primary" in html  # Active button
assert "btn-ghost" in html    # Inactive button
assert "hx-post" in html      # HTMX on inactive

# Test view toggle - list mode active
toggle = render_view_toggle(ViewMode.LIST, "/toggle-view", "#gallery")
html = to_xml(toggle)
# Both buttons should be present
assert "Grid view" in html or "grid_view" in html.lower() or "<svg" in html

print("render_view_toggle tests passed!")

render_view_toggle tests passed!


## Type Filter Buttons

Filter buttons to show/hide specific file types.

In [ ]:
#| export
# Display names for file types
FILE_TYPE_LABELS: dict[FileType, str] = {
    FileType.AUDIO: "Audio",
    FileType.VIDEO: "Video",
    FileType.IMAGE: "Images",
    FileType.DOCUMENT: "Documents",
    FileType.CODE: "Code",
    FileType.DATA: "Data",
    FileType.ARCHIVE: "Archives",
    FileType.OTHER: "Other",
}

In [ ]:
#| export
def render_type_filter_button(
    file_type: FileType,              # File type for this button
    is_active: bool,                  # Whether this type is currently shown
    filter_url: str,                  # URL for filter handler
    hx_target: Optional[str] = None,  # HTMX target for swap
    count: Optional[int] = None,      # Number of files of this type
) -> Button:  # Filter button
    """Render a single type filter button."""
    label = FILE_TYPE_LABELS.get(file_type, file_type.value.title())
    badge_color = MEDIA_TYPE_BADGE_COLORS.get(file_type, str(badge_styles.ghost))
    
    btn_cls = combine_classes(
        btn, btn_sizes.sm,
        btn_colors.primary if is_active else btn_styles.ghost,
        gap(2)
    )
    
    attrs = {
        "cls": btn_cls,
        "hx_post": filter_url,
        "hx_vals": f'{{"file_type": "{file_type.value}", "toggle": "true"}}',
    }
    if hx_target:
        attrs["hx_target"] = hx_target
    attrs["hx_swap"] = "outerHTML"
    
    # Build button content
    content = [
        get_media_type_icon(file_type, size=4, with_color=not is_active),
        Span(label),
    ]
    
    # Add count badge if provided
    if count is not None:
        content.append(
            Span(
                str(count),
                cls=combine_classes(badge, badge_color if is_active else badge_styles.ghost)
            )
        )
    
    return Button(*content, **attrs)

In [ ]:
# Test type filter button
btn_elem = render_type_filter_button(
    FileType.AUDIO, True, "/filter", "#gallery"
)
html = to_xml(btn_elem)
assert "<button" in html
assert "btn-primary" in html  # Active
assert "Audio" in html
assert "hx-post" in html

# Test inactive button
btn_elem = render_type_filter_button(
    FileType.VIDEO, False, "/filter", "#gallery"
)
html = to_xml(btn_elem)
assert "btn-ghost" in html  # Inactive
assert "Video" in html

# Test with count
btn_elem = render_type_filter_button(
    FileType.IMAGE, True, "/filter", "#gallery", count=42
)
html = to_xml(btn_elem)
assert "42" in html
assert "badge" in html

print("render_type_filter_button tests passed!")

render_type_filter_button tests passed!


In [ ]:
#| export
def render_type_filters(
    available_types: List[FileType],  # File types to show buttons for
    active_types: List[FileType],     # Currently active (shown) types
    filter_url: str,                  # URL for filter handler
    hx_target: Optional[str] = None,  # HTMX target for swap
    type_counts: Optional[dict[FileType, int]] = None,  # File counts per type
) -> Any:  # Type filter button group
    """Render type filter buttons."""
    buttons = []
    for file_type in available_types:
        is_active = file_type in active_types
        count = type_counts.get(file_type) if type_counts else None
        buttons.append(
            render_type_filter_button(
                file_type, is_active, filter_url, hx_target, count
            )
        )
    
    return Div(
        *buttons,
        cls=combine_classes(flex_display, flex_wrap.wrap, gap(2))
    )

In [ ]:
# Test type filters
filters = render_type_filters(
    available_types=[FileType.AUDIO, FileType.VIDEO, FileType.IMAGE],
    active_types=[FileType.AUDIO, FileType.IMAGE],
    filter_url="/filter",
    hx_target="#gallery"
)
html = to_xml(filters)
assert "Audio" in html
assert "Video" in html
assert "Images" in html
# Should have 2 active and 1 inactive
assert html.count("btn-primary") == 2
assert html.count("btn-ghost") == 1

# Test with counts
filters = render_type_filters(
    available_types=[FileType.AUDIO, FileType.VIDEO],
    active_types=[FileType.AUDIO],
    filter_url="/filter",
    type_counts={FileType.AUDIO: 10, FileType.VIDEO: 5}
)
html = to_xml(filters)
assert "10" in html
assert "5" in html

print("render_type_filters tests passed!")

render_type_filters tests passed!


## Gallery Controls Bar

Combined controls bar with view toggle and type filters.

In [ ]:
#| export
def render_gallery_controls(
    config: GalleryConfig,            # Gallery configuration
    current_view: ViewMode,           # Current view mode
    active_types: List[FileType],     # Currently active type filters
    toggle_view_url: str,             # URL for view toggle
    filter_url: str,                  # URL for type filter
    hx_target: Optional[str] = None,  # HTMX target
    type_counts: Optional[dict[FileType, int]] = None,  # File counts per type
    controls_id: Optional[str] = None,  # HTML ID for controls container
) -> Any:  # Controls bar component
    """Render the gallery controls bar."""
    controls = []
    
    # View toggle
    if config.allow_view_toggle:
        controls.append(
            render_view_toggle(current_view, toggle_view_url, hx_target)
        )
    
    # Type filters
    if config.filter.allow_type_filter:
        controls.append(
            render_type_filters(
                available_types=config.filter.enabled_types,
                active_types=active_types,
                filter_url=filter_url,
                hx_target=hx_target,
                type_counts=type_counts
            )
        )
    
    container_cls = combine_classes(
        flex_display, flex_wrap.wrap, items.center, justify.between,
        gap(4), p(4),
        bg_dui.base_200,
        rounded.t.lg
    )
    
    attrs = {"cls": container_cls}
    if controls_id:
        attrs["id"] = controls_id
    
    return Div(*controls, **attrs)

In [ ]:
# Test gallery controls
config = GalleryConfig()
controls = render_gallery_controls(
    config=config,
    current_view=ViewMode.GRID,
    active_types=[FileType.AUDIO, FileType.VIDEO, FileType.IMAGE, FileType.DOCUMENT],
    toggle_view_url="/toggle-view",
    filter_url="/filter",
    hx_target="#gallery",
    controls_id="gallery-controls"
)
html = to_xml(controls)
assert 'id="gallery-controls"' in html
assert 'class="join"' in html  # View toggle button group
assert "Audio" in html  # Type filters
assert "Video" in html

# Test with view toggle disabled
config_no_toggle = GalleryConfig(allow_view_toggle=False)
controls = render_gallery_controls(
    config=config_no_toggle,
    current_view=ViewMode.GRID,
    active_types=[FileType.AUDIO],
    toggle_view_url="/toggle-view",
    filter_url="/filter"
)
html = to_xml(controls)
assert 'class="join"' not in html  # No view toggle button group
assert "Audio" in html  # Type filters still present

print("render_gallery_controls tests passed!")

render_gallery_controls tests passed!


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()